In [75]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings("ignore") # to avoid deprecation warnings
from sklearn.model_selection import train_test_split,cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression,LinearRegression, Lasso, Ridge
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)


In [2]:
data=pd.read_csv('Walmart_Store_sales.csv')

In [3]:
data.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858
1,13.0,25-03-2011,1807545.43,0.0,42.38,3.435,128.616064,7.470
2,17.0,27-07-2012,NaN,0.0,NaN,NaN,130.719581,5.936
3,11.0,NaN,1244390.03,0.0,84.57,NaN,214.556497,7.346
4,6.0,28-05-2010,1644470.66,0.0,78.89,2.759,212.412888,7.092


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         150 non-null    float64
 1   Date          132 non-null    object 
 2   Weekly_Sales  136 non-null    float64
 3   Holiday_Flag  138 non-null    float64
 4   Temperature   132 non-null    float64
 5   Fuel_Price    136 non-null    float64
 6   CPI           138 non-null    float64
 7   Unemployment  135 non-null    float64
dtypes: float64(7), object(1)
memory usage: 9.5+ KB


In [5]:
data.describe(include='all')

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000000,132,1.360000e+02,138.000000,132.000000,136.000000,138.000000,135.000000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,19-10-2012,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.866667,NaN,1.249536e+06,0.079710,61.398106,3.320853,179.898509,7.598430
std,6.231191,NaN,6.474630e+05,0.271831,18.378901,0.478149,40.274956,1.577173
min,1.000000,NaN,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000
25%,4.000000,NaN,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500
50%,9.000000,NaN,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000
75%,15.750000,NaN,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000


In [6]:
data.isnull().sum()/data.shape[0]*100

Store            0.000000
Date            12.000000
Weekly_Sales     9.333333
Holiday_Flag     8.000000
Temperature     12.000000
Fuel_Price       9.333333
CPI              8.000000
Unemployment    10.000000
dtype: float64

In [7]:
data['Date'] = pd.to_datetime(data['Date'], format='%d-%m-%Y')


In [8]:

check_outliers_list=['CPI', 'Fuel_Price', 'Unemployment', 'Temperature']
data[check_outliers_list].apply(lambda x: (x > (abs(x.mean()) + 3 * x.std())).sum())


CPI             0
Fuel_Price      0
Unemployment    5
Temperature     0
dtype: int64

In [9]:
px.scatter_matrix(data,width=1000,height=1000).show()

In [10]:
data=data.loc[data['Unemployment']<13]
px.scatter_matrix(data,width=1000,height=1000).show()

In [11]:
data['year'] = data.loc[:,'Date'].dt.year
data['month'] = data.loc[:,'Date'].dt.month
data['day'] = data.loc[:,'Date'].dt.day
data['day_of_week']=data.loc[:,'Date'].dt.dayofweek

In [12]:
data=data.loc[~data['Weekly_Sales'].isnull()]
data.isnull().sum()/data.shape[0]*100

Store            0.000000
Date            12.820513
Weekly_Sales     0.000000
Holiday_Flag     8.547009
Temperature      9.401709
Fuel_Price       9.401709
CPI              7.692308
Unemployment     0.000000
year            12.820513
month           12.820513
day             12.820513
day_of_week     12.820513
dtype: float64

Preprocessing

In [165]:
X=data.drop(columns=['Weekly_Sales','Date'])
y=data['Weekly_Sales']

In [166]:
categorical_features=['Store','Holiday_Flag']
numeric_features= [c for c in X.columns if c not in categorical_features]

print('categorical_features :',categorical_features)
print('numeric_features :',numeric_features)


categorical_features : ['Store', 'Holiday_Flag']
numeric_features : ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'year', 'month', 'day', 'day_of_week']


In [15]:
X.isnull().sum()/len(X)*100

Store            0.000000
Holiday_Flag     8.547009
Temperature      9.401709
Fuel_Price       9.401709
CPI              7.692308
Unemployment     0.000000
year            12.820513
month           12.820513
day             12.820513
day_of_week     12.820513
dtype: float64

In [167]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="mean")),  
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        
        ('OHE',OneHotEncoder(drop="first") )
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [168]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2)

In [169]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)


In [ ]:
X_train[:1]

In [ ]:
X_test[:1]

In [70]:
model=LinearRegression()
model.fit(X_train,Y_train)

LinearRegression()

In [71]:
print('Train Score :',model.score(X_train,Y_train))
print('Test Score :',model.score(X_test,Y_test))

Train Score : 0.9685318932541273
Test Score : 0.9725196029475834


In [72]:
features_names=numeric_features+list(preprocessor.named_transformers_['cat'].named_steps['OHE'].get_feature_names_out())


pd.DataFrame({'features':features_names,'Coefficient':model.coef_,'Intercept':model.intercept_}).sort_values(by='Coefficient',ascending=False)

,features,Coefficient,Intercept
19,Store_14.0,7.283659e+05,1.502868e+06
10,Store_4.0,6.664899e+05,1.502868e+06
16,Store_10.0,5.887909e+05,1.502868e+06
18,Store_13.0,5.272571e+05,1.502868e+06
8,Store_2.0,3.712542e+05,1.502868e+06
25,Store_20.0,3.530071e+05,1.502868e+06
24,Store_19.0,1.248857e+05,1.502868e+06
2,CPI,1.124027e+05,1.502868e+06
5,month,5.681365e+04,1.502868e+06
7,day_of_week,-1.164153e-09,1.502868e+06


In [73]:
# pd.DataFrame({'features':numeric_features+categorical_features,'Coefficient':model.coef_,'Intercept':model.intercept_}).sort_values(by='Coefficient',ascending=False)

In [176]:
model=Ridge()

params={'alpha':list(np.linspace(0,10,100))}
gridsearch=GridSearchCV(model,param_grid=params,cv=10)

In [177]:
gridsearch.fit(X_train,Y_train)
print('Train Score :',gridsearch.score(X_train,Y_train))
print('Test Score :',gridsearch.score(X_test,Y_test))
print('Best Params :',gridsearch.best_params_)
print('Best Score :',gridsearch.best_score_)
print('Best Estimator :',gridsearch.best_estimator_)


Train Score : 0.9830117346704309
Test Score : 0.8569214258270652
Best Params : {'alpha': np.float64(0.0)}
Best Score : 0.9440147832701818
Best Estimator : Ridge(alpha=np.float64(0.0))


In [175]:
features_names=numeric_features+list(preprocessor.named_transformers_['cat'].named_steps['OHE'].get_feature_names_out())


pd.DataFrame({'features':features_names,'Coefficient':gridsearch.best_estimator_.coef_}).sort_values(by='Coefficient',ascending=False)

,features,Coefficient
10,Store_4.0,7.818786e+05
18,Store_13.0,6.816037e+05
16,Store_10.0,6.061223e+05
19,Store_14.0,5.563013e+05
25,Store_20.0,3.964644e+05
8,Store_2.0,2.099705e+05
2,CPI,1.480673e+05
24,Store_19.0,7.951731e+04
5,month,2.720978e+04
12,Store_6.0,2.657869e+04


Selection des 3 meilleurs colonnes et remodelisation

In [181]:
categorical_features=['Store']
numeric_features= [c for c in X.columns if c not in categorical_features]

In [182]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="mean")),  
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        
        ('OHE',OneHotEncoder(drop="first") )
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [183]:
X=data.loc[:,['Store','CPI','month']]
y=data['Weekly_Sales']



X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2)
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)


In [184]:
model=Lasso()
params={'alpha':list(np.linspace(0,10,100))}  
gridsearch=GridSearchCV(model,param_grid=params,cv=10)
gridsearch.fit(X_train,Y_train)
print('Train Score :',gridsearch.score(X_train,Y_train))
print('Test Score :',gridsearch.score(X_test,Y_test))

Train Score : 0.9607888547822887
Test Score : 0.9566282334421866


In [185]:
features_names=numeric_features+list(preprocessor.named_transformers_['cat'].named_steps['OHE'].get_feature_names_out())


pd.DataFrame({'features':features_names,'Coefficient':gridsearch.best_estimator_.coef_}).sort_values(by='Coefficient',ascending=False)

,features,Coefficient
4,Store_4.0,7.981661e+05
13,Store_14.0,5.883122e+05
12,Store_13.0,5.634835e+05
10,Store_10.0,4.308020e+05
2,Store_2.0,4.224587e+05
19,Store_20.0,3.814013e+05
0,CPI,6.251454e+04
1,month,4.882661e+04
6,Store_6.0,-1.830713e+03
18,Store_19.0,-2.925445e+04
